In [0]:
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.erp_product_category")
df.display()

In [0]:
# rename columns
df = df.withColumnRenamed("CAT", "category")\
    .withColumnRenamed("SUBCAT", "sub_category")\
    .withColumnRenamed("ID", "id")\
    .withColumnRenamed("MAINTENANCE", "maintenance")

In [0]:
# normalize df.maintenance
df = df.withColumn("maintenance", 
                   F.when(F.lower(df.maintenance)=="yes", F.lit(True))\
                       .when(F.lower(df.maintenance)=="no", F.lit(False))\
                       .otherwise(F.lit(None))
                   )
df.select("maintenance").distinct().show()

In [0]:
df.printSchema()

In [0]:
# trim() string fields
for col_name, dtype in df.dtypes:
    if dtype == 'string':
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
df.printSchema()

In [0]:
# write table
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_product_category")